In [1]:
%matplotlib qt5
import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import orix
import matplotlib.pyplot as plt

from pathlib import Path
from diffpy.structure import Atom, Lattice, Structure
from diffsims.generators.simulation_generator import SimulationGenerator
from orix.vector import Miller, Vector3d
from orix.plot import IPFColorKeyTSL
from matplotlib.lines import Line2D
from dataclasses import dataclass

import plotly.graph_objects as go

In [2]:
PATH_SPED5 = Path("D:/datasets/250225_Ag_oxww/20250225_175525/SPED5_calibrated.zspy")
PATH_SPED6 = Path("D:/datasets/250225_Ag_oxww/20250225_180346/SPED6_calibrated.zspy")
PATH_SPED7 = Path("D:/datasets/250225_Ag_oxww/20250225_181109/SPED7_calibrated.zspy")
PATH_SPED8 = Path("D:/datasets/250225_Ag_oxww/20250225_182247/SPED8_calibrated.zspy")

path_list = [PATH_SPED6, PATH_SPED7, PATH_SPED8]

colors = ["blue", "red", "green", "orange"]
labels = ["SPED5", "SPED6", "SPED7", "SPED8"]

In [3]:
# Define the crystal structure for silver (Ag)
a = 4.08
atoms = [Atom('Ag', [0, 0, 0]), Atom('Ag', [0.5, 0.5, 0]), Atom('Ag', [0.5, 0, 0.5]), Atom('Ag', [0, 0.5, 0.5])]
lattice = Lattice(a, a, a, 90, 90, 90)
phase = orix.crystal_map.Phase(name='Ag', space_group=225, structure=Structure(atoms, lattice))

angular_resolution = 0.5
grid = orix.sampling.get_sample_reduced_fundamental(angular_resolution, point_group=phase.point_group)
orientations = orix.quaternion.Orientation(grid, symmetry=phase.point_group)

simgen = SimulationGenerator(precession_angle=1., minimum_intensity=1e-5)
simulations = simgen.calculate_diffraction2d(phase=phase, rotation=grid, reciprocal_radius=1.5, with_direct_beam=False, max_excitation_error=0.01)

# Create IPF color keys for x, y, and z directions
key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

In [4]:
def machine_temp(path):
    SPED_data = hs.load(path, lazy=False)
    SPED_data.change_dtype('float32')
    
    ny, nx = SPED_data.axes_manager.signal_shape
    
    center = (ny/2 - 0.5, nx/2 - 0.5)
    SPED_data.calibration(center=center)
    
    s_rad1d = SPED_data.get_azimuthal_integral1d(64)
    s_rad2d = SPED_data.get_azimuthal_integral2d(npt=256)

    s_rad1d.decomposition(True)
    
    elbow = s_rad1d.estimate_elbow_position()

    # Force elbow to be a Python int
    if hasattr(elbow, "compute"):
        elbow = elbow.compute()
    elbow = int(np.asarray(elbow).item())

    s_rad_decomp = s_rad1d.get_decomposition_model(elbow)

    s_rad1d.blind_source_separation(2)
    s_rad1d.cluster_analysis(cluster_source="bss", number_of_components=2, n_clusters=2)

    labels = s_rad1d.get_cluster_labels()
    
    template_results = s_rad2d.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)

    single_phase = template_results.to_single_phase_orientations()[:,:,0]
    xmap = template_results.to_crystal_map()
    oris = xmap.orientations
    corrs = template_results.data[:,:,0,1].flatten()

    return labels, template_results, xmap, oris, corrs, single_phase

@dataclass
class SPEDResult:
    labels: object
    results: object
    xmap: object
    oris: object
    corrs: object
    single_phase: object

In [5]:
sped5 = SPEDResult(*machine_temp(PATH_SPED5))
sped6 = SPEDResult(*machine_temp(PATH_SPED6))
sped7 = SPEDResult(*machine_temp(PATH_SPED7))
sped8 = SPEDResult(*machine_temp(PATH_SPED8))

  0%|          | 0/73 [00:00<?, ?it/s]

  0%|          | 0/73 [00:00<?, ?it/s]

Decomposition info:
  normalize_poissonian_noise=True
  algorithm=SVD
  output_dimension=None
  centre=None


  0%|          | 0/5 [00:00<?, ?it/s]

WARNING | Hyperspy | HyperSpy already performs its own data whitening (whiten_method='PCA'), so it is ignored for algorithm='sklearn_fastica'. (hyperspy.learn.mva:855)
Blind source separation info:
  number_of_components=2
  algorithm=sklearn_fastica
  diff_order=1
  reverse_component_criterion=factors
  whiten_method=PCA
scikit-learn estimator:
FastICA(tol=1e-10, whiten=False)


...

  0%|          | 0/290 [00:00<?, ?it/s]

WARNING | Hyperspy | The function you applied does not take into account the difference of scales in-between axes. (hyperspy.signal:5487)
WARNING | Hyperspy | The function you applied does not take into account the difference of units in-between axes. (hyperspy.signal:5492)


  0%|          | 0/33 [00:00<?, ?it/s]

  0%|          | 0/65 [00:00<?, ?it/s]

  0%|          | 0/65 [00:00<?, ?it/s]

In [17]:
def main_function(data):
    nav_mask = data.labels.inav[0].T
    nav_mask.change_dtype('bool')

    nav_mask2 = data.labels.inav[1].T
    nav_mask2.change_dtype('bool')
    
    phase = data.single_phase[nav_mask]
    phase2 = data.single_phase[nav_mask2]
    fig = phase.scatter(projection="ipf", return_figure=True, label="grain 1")

    phase2.scatter(
        projection="ipf",
        figure=fig,
        c='r',
        label="grain 2"
    )
    plt.legend()
    plt.show()

main_function(sped8)

In [18]:
mask = sped8.labels.inav[0].T
mask.change_dtype('bool')
mask.plot()

In [ ]:
def get_marker_offset_signal(sig):
    '''
    Get the signal positions needed to define markers based on the signal.
    
    Parameters
    ----------
    sig: numpy.ndarray
        The signal in one navigation position, as signal.data

    Returns
    ----------
    xys: numpy.ndarray
        The signal positions as [[xi,yi]] in calibrated x-units.
    
    '''
    xys = np.zeros((len(sig), 2))
    for i, sd in zip(range(len(sig)), sig):
        xys[i] = [i*scale+offset, sd]
    return np.array(xys)

In [ ]:
offsets = s_rad_decomp.map(get_marker_offset_signal, inplace=False, ragged=True)
mark = hs.plot.markers.Points(offsets.data.T, color='blue')

In [ ]:
s_rad.blind_source_separation(2)

In [ ]:
s_rad.plot_bss_results()

In [ ]:
s_rad.cluster_analysis(cluster_source="bss", number_of_components=2, n_clusters=2)

In [ ]:
s_rad.plot_cluster_labels(axes_decor="off");
s_rad.plot_cluster_signals();

In [ ]:
labels = s_rad.get_cluster_labels()
nav_mask = labels.inav[1]
nav_mask.change_dtype('bool')
nav_mask.plot()

In [ ]:
s_max = s.max()
s_max.plot()

In [ ]:
direct_beam_mask0 = s.get_direct_beam_mask(radius=13.)
direct_beam_mask1 = s.get_direct_beam_mask(radius=70.)
direct_beam_mask1.data = ~direct_beam_mask1.data
signal_mask = direct_beam_mask0+direct_beam_mask1
signal_mask.plot()

In [ ]:
s.decomposition(True, signal_mask=signal_mask)